In [1]:
import codecs
import os
import shutil
from PIL import Image
import csv
import torch
from torch.nn.modules import *
from functools import partial
import math
import time
import timm
import random
#文件存为csv
numarr = range(200)
print(numarr)
ran = random.sample(numarr,200)
ran = ran[:40]#随机取20%作为测试集
all_file_dir='./FERET_80_80'
class_list = [c for c in os.listdir(all_file_dir) if os.path.isdir(os.path.join(all_file_dir, c)) and not c.endswith('Set') and not c.startswith('.')]
all_file = open("./trainfile.csv", 'w')
test_file=open("./testfile.csv","w")
with codecs.open(os.path.join(all_file_dir,'labellist.csv'), "w") as label_list:
    label_id = 1
    for class_dir in class_list:#加类别
        csv.writer(label_list).writerow([label_id, class_dir])
        image_path_pre = os.path.join(all_file_dir, class_dir)
        label_id += 1
    label_id = 0
    for class_dir in class_list:
        image_path_pre = os.path.join(all_file_dir, class_dir)
        label_id = int(class_dir[-3:])
        check = 0
        if label_id in ran:
            test_num = random.randint(1,7)
            check = 1
        for file in os.listdir(image_path_pre):#加测试集和训练集
            try:
                img = Image.open(os.path.join(image_path_pre, file))
                shutil.copyfile(os.path.join(image_path_pre, file),  os.path.join(all_file_dir, file))
                numimage = int(file[:2])
                if numimage != test_num or check == 0:
                    csv.writer(all_file).writerow([os.path.join(all_file_dir,os.path.join(class_dir, file)), label_id])
                else:
                    csv.writer(test_file).writerow([os.path.join(all_file_dir,os.path.join(class_dir, file)), label_id])
            except Exception as e:
                pass
all_file.close()
test_file.close()

'numarr = range(200)\nprint(numarr)\nran = random.sample(numarr,200)\nran = ran[:40]\nall_file_dir=\'./FERET_80_80\'\nclass_list = [c for c in os.listdir(all_file_dir) if os.path.isdir(os.path.join(all_file_dir, c)) and not c.endswith(\'Set\') and not c.startswith(\'.\')]\nall_file = open("./trainfile.csv", \'w\')\ntest_file=open("./testfile.csv","w")\nwith codecs.open(os.path.join(all_file_dir,\'labellist.csv\'), "w") as label_list:\n    #加入训练集\n    label_id = 1\n    for class_dir in class_list:\n        csv.writer(label_list).writerow([label_id, class_dir])\n        image_path_pre = os.path.join(all_file_dir, class_dir)\n                # 存在一些文件打不开，此处需要稍作清洗\n        label_id += 1\n    label_id = 0\n    for class_dir in class_list:\n        image_path_pre = os.path.join(all_file_dir, class_dir)\n        label_id = int(class_dir[-3:])\n        check = 0\n        if label_id in ran:\n            test_num = random.randint(1,7)\n            check = 1\n        for file in os.listdir(image_

In [2]:
import pandas as pd
import sklearn
import numpy as np
import cv2
import csv
def loadimages(): #读图片
    train_images = []
    train_labels = []
    with open('trainfile.csv','r') as csvfile:
        read = csv.reader(csvfile)
        train_data = [row for row in read]
    train_data = sklearn.utils.shuffle(train_data) #打乱
    #读取
    for i in train_data:
        img = cv2.imread(i[0],0) 
        train_images.append(img)
        train_labels.append(int(i[1]))
    train_images = np.asarray(train_images)
    train_labels =np.asarray(train_labels)
    test_images = []
    test_labels = []
    with open('testfile.csv','r') as csvfile:
        reada = csv.reader(csvfile)
        test_data = [row for row in reada]
    test_data = sklearn.utils.shuffle(test_data) #打乱
    #读取
    for i in test_data:
        img = cv2.imread(i[0],0) 
        test_images.append(img)
        test_labels.append(int(i[1]))
    test_images = np.asarray(test_images)
    test_labels =np.asarray(test_labels)
    return train_images,train_labels,test_images,test_labels


In [34]:
from sklearn import neighbors
import tkinter
from tkinter import filedialog
import matplotlib as plt
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
class face(object):
    def __init__(self,dimnum=150,n_neighbors=5,dsize=(80,80)):
        '''
        dimnum:pca降维后的维度
        neighbor：knn参数
        dsize：图像预处理的尺寸
        '''
        self._dimnum = dimnum
        self._dsize = dsize
        self._mean = 0.0
        self._knn = neighbors.KNeighborsClassifier(n_neighbors)       
        self.pca = sklearn.decomposition.PCA(dimnum)
            
    def _pca(self,x):
        x=x.T
        mean = np.reshape(np.mean(x,axis=1),(-1,1))  #求均值
        self._mean = mean
        print(mean.shape)
        print(x.shape)
        diff = x - mean
        cov = np.dot(diff.T,diff)/diff.shape[1] #协方差
        eigvals,eigvects = np.linalg.eig(cov) #特征值和特征向量
        eigvects = np.dot(diff,eigvects)
        print("特征向量的维度:",eigvects.shape)
        eigvalindex = np.argsort(eigvals) #取最大的dimnum个
        eigvalindex = eigvalindex[::-1]
        eigvalindex = eigvalindex[:self._dimnum]
        eigvects = eigvects/np.linalg.norm(eigvects,axis=0) #归一化
        transmat = (eigvects.T)[eigvalindex,:]
        lowmat = np.dot(transmat,diff)
        lowmat = lowmat.T
        print('降维后的矩阵维度：',lowmat.shape)
        return lowmat,transmat
    def _prepare(self,images):
        new_images = []
        for image in images:
            hist_img = image.flatten() #转为一行
            new_images.append(hist_img)
        new_images = np.asarray(new_images)#列表变为数组
        #new_images = StandardScaler().fit_transform(new_images)
        print(new_images.shape)
        return new_images
    def fit(self,x_train,y_train):
        '''
        x_train训练集数据
        y_train训练集标签
        '''
        x_train = self._prepare(x_train)
        x_train_pca,self._transmat = self._pca(x_train)
        #x_t= np.array(x_train_pca)
        #x_t = x_train
        x_t = self.pca.fit_transform(x_train)
        y_t = np.array(y_train)
        np.array(x_t).astype(float)
        print(y_t)
        #self._knn.fit(x_t.astype('float'),y_t.astype('int'))
        self._knn.fit(x_train_pca.astype('float'),y_t.astype('int'))
    def predict(self,x_test,y_test):
        x_test = self._prepare(x_test)
        #x_test = self.pca.transform(x_test)
        x_test_pca = np.dot(self._transmat,x_test.T-self._mean)
        x_test_pca = x_test_pca.T
        #result = self._knn.predict(x_test)
        result = self._knn.predict(x_test_pca.astype('float'))
        print(result)
        print(y_test)
        sum = 0
        for i in range(len(result)):
            if result[i] == y_test[i]:
                sum+= 1
        scores = sum / len(result)
        return scores

In [47]:
if __name__=='__main__':
    Face = face(15,n_neighbors=1)
    x_train,y_train,x_test,y_test = loadimages()
    Face.fit(x_train,y_train)
    y_pred = Face.predict(x_test,y_test)
    print(y_pred)

(1360, 6400)
(6400, 1)
(6400, 1360)
特征向量的维度: (6400, 1360)
降维后的矩阵维度： (1360, 15)
[ 83 103 152 ... 191  69   9]
436
(40, 6400)
[ 92   2 101 154   3  88  56 184  22   2 157  34  84 130  56  15 189 179
 132  88 173  84  37 150 123 123  39 108  96 111 195 129  86 172 183   1
 140 100  55 145]
[ 92 142 114 154  63  88  50 166  21 196 157 128 112 130  56  15 177 136
 153  22 173 103  37  32   9 144  39  43  96  79 176 151  86  87 183   1
 140 113  49 145]
0.4


d:\Anaconda3\envs\pytorch\lib\site-packages\ipykernel_launcher.py:63: ComplexWarning: Casting complex values to real discards the imaginary part
d:\Anaconda3\envs\pytorch\lib\site-packages\ipykernel_launcher.py:70: ComplexWarning: Casting complex values to real discards the imaginary part
